# Adım 7 — Saldırı ailesi deneyi (görülmemiş saldırı TÜRÜ)

**Soru:** Bildirideki açık soru şuydu: görülmemiş veri setinde başarı düşüyor, ama bu **yeni saldırı türünden** mi, yoksa **veri setinin yazılış tarzından** mı?

**Fikir:** Bu defterde model **sadece wambosec** ile eğitilir. Böylece veri seti tarzı hep aynı kalır. Her koşuda wambosec'in 14 saldırı ailesinden **biri** eğitimden tamamen çıkarılır ve model o aileyle sınanır.
- Model, çıkarılan aileyi de iyi yakalıyorsa → düşüşün asıl sebebi büyük ihtimalle **veri seti tarzı**.
- Çıkarılan aileyi yakalayamıyorsa → **yeni saldırı türü** de önemli bir sebep.

**Ayarlar:** 05 ile aynı (3 tur, parti 16, öğrenme hızı 2e-5, en fazla 128 token, tohum 42).

**Nasıl çalıştırılır:**
1. `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`.
2. `Çalışma zamanı → Tümünü çalıştır`. Drive izni sorarsa izin ver (iki kez sorabilir).
3. **Sekmeyi kapatma, bilgisayar uykuya geçmesin.** Toplam süre: BERT için yaklaşık 30-40 dakika (15 koşu).
4. Kopsa bile sorun yok: tekrar `Tümünü çalıştır` de, biten koşuları atlar.
5. Bitince en alttaki **ÖZET** hücresi sonuç tablosunu yazdırır.

In [ ]:
!pip -q install transformers scikit-learn

In [ ]:
import os, time, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GroupShuffleSplit

# ---- Ayarlar (Karar 8) ----
TOHUM = 42            # her koşuda aynı rastgelelik
TUR = 3               # veriyi kaç kez baştan sona gezecek
PARTI = 16            # aynı anda kaç örnek işlenecek
OGRENME_HIZI = 2e-5   # her adımda ne kadar düzeltme (küçük tutulur)
MAKS_UZUNLUK = 128    # uzun metinler bu kadar parçada kesilir (bildiride sınırlılık olarak yazılacak)
HEDEF_FPR = 0.01      # %1 yanlış pozitif oranı

cihaz = "cuda" if torch.cuda.is_available() else "cpu"
KARISIK = (cihaz == "cuda")   # GPU'da hızlandırmak için yarı hassasiyet (fp16)
print("Kullanılan cihaz:", cihaz)
if cihaz == "cpu":
    print("UYARI: GPU yok! Çalışma zamanı türünden T4 GPU seçmelisin, yoksa eğitim çok yavaş olur.")
else:
    print("Kart:", torch.cuda.get_device_name(0))


def tohum_ayarla(t=TOHUM):
    random.seed(t); np.random.seed(t); torch.manual_seed(t)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(t)

## Drive'ı bağla (sonuçlar kaybolmasın)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SONUC_KLASORU = "/content/drive/MyDrive/YiitliProje_sonuclar"
os.makedirs(SONUC_KLASORU, exist_ok=True)
AILE_DOSYASI = SONUC_KLASORU + "/aile_sonuclari.csv"        # her koşunun özeti
REF_DOSYASI = SONUC_KLASORU + "/aile_referans_yakalama.csv"  # tüm aileler görülünce aile bazında yakalama
print("Sonuçlar:", AILE_DOSYASI)

## veri_birlesik.csv'yi indir

In [ ]:
from google.colab import auth
auth.authenticate_user()                       # Google hesabınla giriş
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

DOSYA_ID = "BURAYA_DRIVE_DOSYA_NUMARASI"   # veri_birlesik.csv'nin Drive numarası
servis = build("drive", "v3")
istek = servis.files().get_media(fileId=DOSYA_ID)
with open("veri_birlesik.csv", "wb") as f:
    indirici = MediaIoBaseDownload(f, istek)
    bitti = False
    while not bitti:
        _, bitti = indirici.next_chunk()
print("İndirildi. Boyut (bayt):", os.path.getsize("veri_birlesik.csv"))

In [ ]:
df = pd.read_csv("veri_birlesik.csv")
print("Şekil:", df.shape, "(beklenen: 23081 x 7)")
print("Sütunlar:", list(df.columns))
print()
print(df.groupby(["rol", "kaynak", "bolum"]).agg(satir=("label", "size"), zararli=("label", "sum")))

# Basit güvenlik kontrolleri: beklediğimiz değerler yoksa burada dururuz
beklenen = {"safeguard", "neuralchemy", "wambosec", "jackhhao", "deepset"}
assert beklenen <= set(df["kaynak"]), f"Beklenen kaynak adları yok! Bulunan: {set(df['kaynak'])}"
assert set(df["bolum"]) <= {"train", "val", "test"}, f"Beklenmeyen bölüm adı: {set(df['bolum'])}"
assert set(df["label"]) <= {0, 1}, "label sadece 0 ve 1 olmalı"
print("\nBoş metin sayısı:", df["text"].isna().sum())
df = df.dropna(subset=["text"]).reset_index(drop=True)
df["text"] = df["text"].astype(str)

## wambosec saldırı aileleri (04'teki kuralların aynısı)

In [ ]:
import re
def kategori_temizle(k):
    k = str(k).lower().strip()
    k = re.sub(r"^tier\s*\d+\s*-\s*", "", k)
    return re.sub(r"[\s_/\-&]+", " ", k).strip()

AILE_KURALLARI = [
    ("steganografi",         ["stegan", "whitespace", "first letter", "acrostic", "zero width"]),
    ("unicode_homoglif",     ["unicode", "homoglyph", "fullwidth", "rtl", "combining", "confusable", "bidi", "braille", "zalgo"]),
    ("ascii_art",            ["ascii art"]),
    ("kodlama",              ["encoding", "base64", "hex", "rot13", "url", "html entit", "punycode", "nested", "binary", "leetspeak"]),
    ("bozuk_sozdizimi",      ["malformed", "broken", "unclosed", "incomplete"]),
    ("bicim_dizgisi",        ["format string", "format ansi", "ansi"]),
    ("dil_karisimi",         ["language", "romanized"]),
    ("dilbilimsel_hile",     ["linguistic"]),
    ("anlamsal_gizleme",     ["semantic"]),
    ("baglam_manipulasyonu", ["context"]),
    ("jailbreak_kaliplari",  ["jailbreak"]),
    ("sosyal_muhendislik",   ["social"]),
    ("yetki_taklidi",        ["authority"]),
    ("teknik_gizleme",       ["technical", "obfuscation"]),
]

def aile_ata(satir):
    if satir["label"] == 0:
        return "yok"
    for ad, anahtarlar in AILE_KURALLARI:
        if any(a in kategori_temizle(satir["kategori"]) for a in anahtarlar):
            return ad
    return "diger"

W = df[df["kaynak"] == "wambosec"].copy()
W["aile"] = W.apply(aile_ata, axis=1)
AILELER = [a for a, _ in AILE_KURALLARI]          # 14 aile ("diger" deneye girmez, 38 satır)
print(pd.crosstab(W["aile"], W["bolum"], margins=True))
assert len(W) == 5766, "wambosec 5766 satır olmalı"

## Yardımcı fonksiyonlar (05 ile aynı)

In [ ]:
def egitim_ayir(havuz):
    """Havuzu %90 eğitim / %10 eşik-seçme setine böler (grup bütünlüğünü korur)."""
    havuz = havuz.reset_index(drop=True)
    # grup boşsa her satır kendi başına bir grup sayılır
    gruplar = [f"tek_{i}" if pd.isna(g) else str(g) for i, g in enumerate(havuz["grup"])]
    ayirici = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=TOHUM)
    egitim_idx, esik_idx = next(ayirici.split(havuz, groups=gruplar))
    return havuz.iloc[egitim_idx].reset_index(drop=True), havuz.iloc[esik_idx].reset_index(drop=True)


def esik_sec(y_esik, p_esik, hedef_fpr=HEDEF_FPR):
    """Masumların (label=0) en fazla hedef_fpr kadarı eşiğin üstünde kalacak şekilde eşik seç.
    Karar kuralı: olasılık > eşik ise 'zararlı'."""
    masum = np.sort(p_esik[y_esik == 0])
    k = int(np.floor(hedef_fpr * len(masum)))   # izin verilen yanlış engelleme sayısı
    return float(masum[len(masum) - 1 - k])

In [ ]:
def modeli_yukle(model_adi):
    tokenizer = AutoTokenizer.from_pretrained(model_adi)
    model = AutoModelForSequenceClassification.from_pretrained(model_adi, num_labels=2)
    return tokenizer, model.to(cihaz)


def kodla(tokenizer, metinler):
    k = tokenizer(list(metinler), padding="max_length", truncation=True,
                  max_length=MAKS_UZUNLUK, return_tensors="pt")
    return k["input_ids"], k["attention_mask"]


def egit(model, tokenizer, egitim_df):
    ids, maske = kodla(tokenizer, egitim_df["text"])
    etiket = torch.tensor(egitim_df["label"].astype(int).values)
    uretec = torch.Generator().manual_seed(TOHUM)
    yukleyici = DataLoader(TensorDataset(ids, maske, etiket), batch_size=PARTI,
                           shuffle=True, generator=uretec)
    optimizer = torch.optim.AdamW(model.parameters(), lr=OGRENME_HIZI)
    olcekleyici = torch.amp.GradScaler("cuda", enabled=KARISIK)
    for tur in range(TUR):
        model.train(); toplam = 0.0
        for i, m, e in yukleyici:
            i, m, e = i.to(cihaz), m.to(cihaz), e.to(cihaz)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=KARISIK):
                cikti = model(input_ids=i, attention_mask=m, labels=e)   # tahmin + hata
            optimizer.zero_grad()
            olcekleyici.scale(cikti.loss).backward()   # hatanın nereden geldiğini hesapla
            olcekleyici.step(optimizer)                # modeli düzelt
            olcekleyici.update()
            toplam += cikti.loss.item()
        print(f"  Tur {tur+1}/{TUR} | ortalama loss: {toplam/len(yukleyici):.4f}")


def olasilik_tahmin(model, tokenizer, metinler, parti=64):
    """Her metin için 'zararlı olma olasılığını' (0-1) döndürür. Eğitim yapmaz."""
    model.eval()
    ids, maske = kodla(tokenizer, metinler)
    sonuc = []
    with torch.no_grad():
        for b in range(0, len(ids), parti):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=KARISIK):
                mantik = model(input_ids=ids[b:b+parti].to(cihaz),
                               attention_mask=maske[b:b+parti].to(cihaz)).logits
            sonuc.append(torch.softmax(mantik.float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(sonuc)

## Bir koşu
- **Eğitim havuzu:** wambosec `train` bölümü, çıkarılan ailenin TÜM satırları hariç. Havuzun %10'u eşik seçmeye ayrılır.
- **Görülmeyen aile:** çıkarılan ailenin tüm satırları (train + test), model hiç görmedi.
- **Görülen aileler:** diğer ailelerin `test` bölümündeki saldırılar.
- **Masumlar:** wambosec `test` bölümündeki masum metinler (yanlış alarm için).
- `aile=None` → **referans**: hiçbir aile çıkarılmaz; her ailenin `test` bölümünde yakalama ayrı ayrı yazılır.

In [ ]:
def yakala(p, esik):
    p = np.asarray(p)
    return float((p >= 0.5).mean()), float((p > esik).mean())

def aile_kosusu(model_adi, aile):
    print("=" * 60); print("KOŞU:", model_adi, "| çıkarılan aile:", aile or "YOK (referans)")
    t0 = time.time(); tohum_ayarla()
    havuz = W[W["bolum"] == "train"]
    if aile:
        havuz = havuz[havuz["aile"] != aile]
    egitim, esik_seti = egitim_ayir(havuz)
    tokenizer, model = modeli_yukle(model_adi)
    egit(model, tokenizer, egitim)
    esik = esik_sec(esik_seti["label"].values, olasilik_tahmin(model, tokenizer, esik_seti["text"]))
    test = W[W["bolum"] == "test"]
    p_test = olasilik_tahmin(model, tokenizer, test["text"])
    masum = test["label"].values == 0
    fpr05, fpr_s = yakala(p_test[masum], esik)
    sure = round((time.time() - t0) / 60, 1)
    if aile is None:
        satirlar = []
        for a in AILELER:
            m = (test["aile"] == a).values
            r05, rs = yakala(p_test[m], esik)
            satirlar.append({"model": model_adi, "aile": a, "n_test": int(m.sum()),
                             "yakalama_05": r05, "yakalama_sabit": rs})
        pd.DataFrame(satirlar).to_csv(REF_DOSYASI, mode="a", header=not os.path.exists(REF_DOSYASI), index=False)
        gorulmeyen = {"n": 0, "yak05": float("nan"), "yaks": float("nan")}
        gorulen_m = (test["label"] == 1).values
    else:
        hedef = W[W["aile"] == aile]
        p_h = olasilik_tahmin(model, tokenizer, hedef["text"])
        r05, rs = yakala(p_h, esik)
        gorulmeyen = {"n": len(hedef), "yak05": r05, "yaks": rs}
        gorulen_m = ((test["label"] == 1) & (test["aile"] != aile)).values
    g05, gs = yakala(p_test[gorulen_m], esik)
    satir = pd.DataFrame([{
        "model": model_adi, "cikarilan_aile": aile or "YOK", "n_gorulmeyen": gorulmeyen["n"],
        "gorulmeyen_yakalama_05": gorulmeyen["yak05"], "gorulmeyen_yakalama_sabit": gorulmeyen["yaks"],
        "gorulen_yakalama_05": g05, "gorulen_yakalama_sabit": gs,
        "masum_yanlis_alarm_05": fpr05, "masum_yanlis_alarm_sabit": fpr_s,
        "esik": esik, "n_esik_masum": int((esik_seti["label"] == 0).sum()), "sure_dk": sure}])
    satir.to_csv(AILE_DOSYASI, mode="a", header=not os.path.exists(AILE_DOSYASI), index=False)
    print(satir.round(3).to_string(index=False))
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None

def bitti_mi(model_adi, aile):
    if not os.path.exists(AILE_DOSYASI):
        return False
    s = pd.read_csv(AILE_DOSYASI)
    return ((s["model"] == model_adi) & (s["cikarilan_aile"] == (aile or "YOK"))).any()

## TÜM KOŞULAR (BERT): 1 referans + 14 aile

In [ ]:
MODEL = "bert-base-uncased"
for aile in [None] + AILELER:
    if bitti_mi(MODEL, aile):
        print("ATLANDI (bitmiş):", aile or "YOK"); continue
    aile_kosusu(MODEL, aile)
print("BERT bitti.")

## ÖZET (sonuç tablosu)

In [ ]:
s = pd.read_csv(AILE_DOSYASI).drop_duplicates(subset=["model", "cikarilan_aile"], keep="last")
r = pd.read_csv(REF_DOSYASI).drop_duplicates(subset=["model", "aile"], keep="last")
ozet = s[s["cikarilan_aile"] != "YOK"].merge(
    r.rename(columns={"aile": "cikarilan_aile", "yakalama_05": "referans_yakalama_05", "n_test": "n_ref_test"})[
        ["model", "cikarilan_aile", "referans_yakalama_05", "n_ref_test"]],
    on=["model", "cikarilan_aile"], how="left")
ozet["fark_05"] = ozet["gorulmeyen_yakalama_05"] - ozet["referans_yakalama_05"]
kolon = ["model", "cikarilan_aile", "n_gorulmeyen", "referans_yakalama_05", "gorulmeyen_yakalama_05", "fark_05",
         "gorulmeyen_yakalama_sabit", "gorulen_yakalama_05", "masum_yanlis_alarm_05", "masum_yanlis_alarm_sabit"]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
print(ozet[kolon].sort_values("fark_05").round(3).to_string(index=False))
print()
print("Referans (hiç aile çıkarılmadı):")
print(s[s["cikarilan_aile"] == "YOK"].round(3).to_string(index=False))
print()
print("Ortalama görülmeyen aile yakalama (0,5):", round(ozet["gorulmeyen_yakalama_05"].mean(), 3),
      "| ortalama referans:", round(ozet["referans_yakalama_05"].mean(), 3))

## (İsteğe bağlı) Aynı deney RoBERTa ile — sadece vaktin varsa
Yaklaşık 30-40 dakika daha. Bittikten sonra ÖZET hücresini tekrar çalıştır.

In [ ]:
MODEL = "roberta-base"
for aile in [None] + AILELER:
    if bitti_mi(MODEL, aile):
        print("ATLANDI (bitmiş):", aile or "YOK"); continue
    aile_kosusu(MODEL, aile)
print("RoBERTa bitti.")